# Feature Engineering and Feature Store Setup

This notebook performs feature engineering on the combined emotion dataset and stores the features in SageMaker Feature Store.

**Tasks:**
1. Load and clean combined emotion data (12,803 samples)
2. Standardize labels and fix data quality issues
3. Engineer new features and apply scaling
4. Split data: 40% train, 10% test, 10% validation, 40% production
5. Create SageMaker Feature Group with online/offline stores
6. Ingest training data into Feature Store
7. Validate Feature Store access and functionality

## Setup and Imports

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import json
import pickle
from datetime import datetime
from time import gmtime, strftime, time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.session import Session

print(f"SageMaker version: {sagemaker.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Initialize SageMaker session and S3 client
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
s3_client = boto3.client('s3', region_name=region)

print(f"Region: {region}")
print(f"Bucket: {bucket}")
print(f"Role: {role}")

In [ ]:
# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# S3 paths
s3_input_data_path = f"s3://{bucket}/datalake/aai-540-group-7/Combined_Emotion_Data.csv"
s3_features_prefix = f"s3://{bucket}/features"
s3_artifacts_prefix = f"s3://{bucket}/models/artifacts"
s3_feature_store_prefix = f"s3://{bucket}/feature-store"

print(f"Input data: {s3_input_data_path}")
print(f"Features prefix: {s3_features_prefix}")
print(f"Artifacts prefix: {s3_artifacts_prefix}")

## Phase 1: Data Preprocessing & Feature Engineering

### 1.1 Load and Clean Data

In [ ]:
# Load combined emotion data from S3
print(f"Loading data from {s3_input_data_path}...")
df = pd.read_csv(s3_input_data_path)

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Check for missing values
missing_counts = df.isnull().sum()
print("Missing values per column:")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "No missing values")

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

In [ ]:
# Examine label distribution BEFORE cleaning
print("Label distribution BEFORE standardization:")
label_counts = df['label'].value_counts()
print(label_counts)
print(f"\nUnique labels: {df['label'].unique()}")

In [ ]:
# Fix label inconsistency: standardize all labels to title case
# This fixes 'sad' (5 samples) vs 'Sad' (2,167 samples) issue
df['label'] = df['label'].str.title()

print("Label distribution AFTER standardization:")
label_counts_clean = df['label'].value_counts()
print(label_counts_clean)
print(f"\nTotal samples: {len(df)}")

In [ ]:
# Add required Feature Store columns
# Record ID: unique identifier for each sample
df['record_id'] = df.apply(lambda row: f"emotion_{row.name:06d}", axis=1)

# Event time: timestamp in ISO format (required for Feature Store)
current_time = datetime.now().isoformat()
df['event_time'] = pd.to_datetime('now').isoformat()

print(f"Added record_id and event_time columns")
print(f"Sample record_id: {df['record_id'].iloc[0]}")
print(f"Sample event_time: {df['event_time'].iloc[0]}")

### 1.2 Feature Engineering

In [ ]:
# Create derived features
print("Creating derived features...")

# Frequency range: fundamental frequency variation
df['freq_range'] = df['maxfun'] - df['minfun']

# Spectral contrast: energy difference between high and low frequencies
df['spectral_contrast'] = df['q75'] - df['q25']

# Energy ratio: dominance energy concentration
df['energy_ratio'] = df['meandom'] / (df['maxdom'] + 1e-6)

print("Derived features created:")
print("- freq_range: Fundamental frequency range")
print("- spectral_contrast: Spectral energy contrast")
print("- energy_ratio: Dominance energy ratio")

# Check for NaN/Inf in new features
print(f"\nNaN count in new features:")
print(f"  freq_range: {df['freq_range'].isna().sum()}")
print(f"  spectral_contrast: {df['spectral_contrast'].isna().sum()}")
print(f"  energy_ratio: {df['energy_ratio'].isna().sum()}")

print(f"\nInf count in new features:")
print(f"  freq_range: {np.isinf(df['freq_range']).sum()}")
print(f"  spectral_contrast: {np.isinf(df['spectral_contrast']).sum()}")
print(f"  energy_ratio: {np.isinf(df['energy_ratio']).sum()}")

In [ ]:
# Define feature columns for scaling
# Original acoustic features (20 features)
original_features = [
    'meanfreq', 'sd', 'median', 'q25', 'q75', 'iqr', 'skew', 'kurt',
    'sp_ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun',
    'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx'
]

# Derived features (3 features)
derived_features = ['freq_range', 'spectral_contrast', 'energy_ratio']

# All features to scale
features_to_scale = original_features + derived_features

print(f"Total features to scale: {len(features_to_scale)}")
print(f"  Original features: {len(original_features)}")
print(f"  Derived features: {len(derived_features)}")

In [ ]:
# Apply StandardScaler to normalize features
print("Applying StandardScaler to features...")
scaler = StandardScaler()

# Fit scaler and transform features
df_scaled = df.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Feature scaling complete")

# Verify scaling (mean ≈ 0, std ≈ 1)
print("\nScaled feature statistics (first 5 features):")
print(df_scaled[features_to_scale[:5]].describe())

In [ ]:
# Create label encoding
print("Creating label encoding...")
label_encoder = LabelEncoder()
df_scaled['label_encoded'] = label_encoder.fit_transform(df_scaled['label'])

# Show label mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("\nLabel mapping:")
for label, code in sorted(label_mapping.items(), key=lambda x: x[1]):
    count = (df_scaled['label'] == label).sum()
    print(f"  {code}: {label:12s} ({count:5d} samples)")

### 1.3 Feature Validation

In [ ]:
# Check for NaN/Inf values after scaling
print("Validating scaled features...")
nan_count = df_scaled[features_to_scale].isnull().sum().sum()
inf_count = np.isinf(df_scaled[features_to_scale]).sum().sum()

print(f"\nNaN values in scaled features: {nan_count}")
print(f"Inf values in scaled features: {inf_count}")

if nan_count > 0 or inf_count > 0:
    print("\n[WARNING] WARNING: Found NaN or Inf values after scaling!")
else:
    print("\n[OK] All features are valid (no NaN or Inf values)")

In [ ]:
# Verify feature distributions are normalized
print("Feature distribution statistics (should have mean≈0, std≈1):")
stats = df_scaled[features_to_scale].agg(['mean', 'std'])
print(stats.T.head(10))

# Check if normalization is correct
mean_check = (stats.loc['mean'].abs() < 1e-10).all()
std_check = ((stats.loc['std'] - 1.0).abs() < 1e-10).all()

if mean_check and std_check:
    print("\n[OK] Feature scaling validated: mean≈0, std≈1")
else:
    print("\n[WARNING] WARNING: Feature scaling may not be perfect (this is usually okay)")

## Phase 2: Data Splitting (Stratified)

Split data into:
- **40% Training**: For model training
- **10% Test**: For final model evaluation
- **10% Validation**: For hyperparameter tuning
- **40% Production**: For simulating production inference

In [ ]:
# Stratified splitting to maintain class distribution
print("Performing stratified data splitting...")
print(f"Total samples: {len(df_scaled)}")

# First split: 40% production holdout
train_val_test, production = train_test_split(
    df_scaled,
    test_size=0.40,
    stratify=df_scaled['label'],
    random_state=RANDOM_SEED
)

print(f"After first split:")
print(f"  Train+Val+Test: {len(train_val_test)} (60%)")
print(f"  Production: {len(production)} (40%)")

In [ ]:
# Second split: from remaining 60%, get 40% train (66.7% of 60%)
train, temp = train_test_split(
    train_val_test,
    test_size=0.333,
    stratify=train_val_test['label'],
    random_state=RANDOM_SEED
)

print(f"After second split:")
print(f"  Train: {len(train)} (40% of total)")
print(f"  Test+Val: {len(temp)} (20% of total)")

In [ ]:
# Third split: split temp into test (10%) and validation (10%)
test, validation = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp['label'],
    random_state=RANDOM_SEED
)

print(f"Final split sizes:")
print(f"  Train: {len(train):5d} ({len(train)/len(df_scaled)*100:.1f}%)")
print(f"  Test: {len(test):5d} ({len(test)/len(df_scaled)*100:.1f}%)")
print(f"  Validation: {len(validation):5d} ({len(validation)/len(df_scaled)*100:.1f}%)")
print(f"  Production: {len(production):5d} ({len(production)/len(df_scaled)*100:.1f}%)")
print(f"  Total: {len(train) + len(test) + len(validation) + len(production)}")

### 2.1 Verify Stratification

In [ ]:
# Verify class distribution is maintained across splits
print("Class distribution verification:")
print("\nOriginal dataset:")
print(df_scaled['label'].value_counts(normalize=True).sort_index())

print("\nTraining set:")
print(train['label'].value_counts(normalize=True).sort_index())

print("\nTest set:")
print(test['label'].value_counts(normalize=True).sort_index())

print("\nValidation set:")
print(validation['label'].value_counts(normalize=True).sort_index())

print("\nProduction set:")
print(production['label'].value_counts(normalize=True).sort_index())

### 2.2 Save Split Datasets

In [ ]:
# Save datasets to S3 in both CSV and Parquet formats
print("Saving split datasets to S3...")

splits = {
    'train': train,
    'test': test,
    'validation': validation,
    'production': production
}

for split_name, split_df in splits.items():
    # CSV format
    csv_path = f"{s3_features_prefix}/{split_name}/{split_name}_features.csv"
    split_df.to_csv(csv_path, index=False)
    print(f"  Saved {split_name} CSV: {csv_path}")
    
    # Parquet format (better performance)
    parquet_path = f"{s3_features_prefix}/{split_name}/{split_name}_features.parquet"
    split_df.to_parquet(parquet_path, index=False)
    print(f"  Saved {split_name} Parquet: {parquet_path}")

print("\n[OK] All datasets saved successfully")

### 2.3 Create Production Batches (Optional)

In [ ]:
# Split production data into batches to simulate production traffic over time
# This is useful for demonstrating model monitoring and data drift detection
num_batches = 5
batch_size = len(production) // num_batches

print(f"Creating {num_batches} production batches...")
print(f"Batch size: ~{batch_size} samples per batch")

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = start_idx + batch_size if i < num_batches - 1 else len(production)
    batch_df = production.iloc[start_idx:end_idx]
    
    batch_path = f"{s3_features_prefix}/production/batches/production_batch_{i+1:03d}.csv"
    batch_df.to_csv(batch_path, index=False)
    print(f"  Batch {i+1}: {len(batch_df)} samples -> {batch_path}")

print("\n[OK] Production batches created")

### 2.4 Save Split Indices for Reproducibility

In [ ]:
# Save split indices to enable reproducibility
split_indices = {
    'train': train.index.tolist(),
    'test': test.index.tolist(),
    'validation': validation.index.tolist(),
    'production': production.index.tolist(),
    'random_seed': RANDOM_SEED,
    'split_timestamp': datetime.now().isoformat()
}

indices_path = f"{s3_features_prefix}/split_indices.json"
with open('/tmp/split_indices.json', 'w') as f:
    json.dump(split_indices, f, indent=2)

# Upload to S3
s3_client.upload_file('/tmp/split_indices.json', bucket, indices_path.replace(f"s3://{bucket}/", ""))
print(f"Split indices saved to: {indices_path}")

## Phase 3: SageMaker Feature Store Setup

### 3.1 Prepare Data for Feature Store

In [ ]:
# Prepare training data for Feature Store ingestion
# Feature Store requires specific column types and formats
feature_store_df = train.copy()

# Ensure event_time is in correct format (fractional seconds)
feature_store_df['event_time'] = pd.to_datetime(feature_store_df['event_time']).astype('int64') // 10**9
feature_store_df['event_time'] = feature_store_df['event_time'].astype('float64')

# Convert record_id to string (required for Feature Store)
feature_store_df['record_id'] = feature_store_df['record_id'].astype('string')

# Convert label to string
feature_store_df['label'] = feature_store_df['label'].astype('string')

# Convert s3_key_s to string if it exists
if 's3_key_s' in feature_store_df.columns:
    feature_store_df['s3_key_s'] = feature_store_df['s3_key_s'].astype('string')

print(f"Feature Store dataframe prepared: {feature_store_df.shape}")
print(f"\nColumn types:")
print(feature_store_df.dtypes)

### 3.2 Create Feature Group

In [ ]:
# Create Feature Group
feature_group_name = "emotion-audio-features"

print(f"Creating Feature Group: {feature_group_name}")

feature_group = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=sess
)

print(f"Feature Group object created: {feature_group.name}")

In [ ]:
# Load feature definitions from dataframe
print("Loading feature definitions...")
feature_group.load_feature_definitions(data_frame=feature_store_df)

print(f"\nFeature definitions loaded: {len(feature_group.feature_definitions)} features")
print("\nFirst 10 features:")
for i, feature_def in enumerate(feature_group.feature_definitions[:10]):
    print(f"  {i+1}. {feature_def['FeatureName']:20s} ({feature_def['FeatureType']})")

### 3.3 Create Feature Group with Online/Offline Stores

In [ ]:
# Create Feature Group with both online and offline stores
print(f"Creating Feature Group with online and offline stores...")
print(f"This may take 1-2 minutes...\n")

try:
    feature_group.create(
        s3_uri=f"{s3_feature_store_prefix}/emotion-features",
        record_identifier_name="record_id",
        event_time_feature_name="event_time",
        role_arn=role,
        enable_online_store=True,
        description="Emotion classification features with audio acoustic properties"
    )
    
    print("[OK] Feature Group created successfully")
    print(f"\nFeature Group ARN: {feature_group.describe()['FeatureGroupArn']}")
    
except Exception as e:
    if 'ResourceInUse' in str(e):
        print(f"[WARNING] Feature Group '{feature_group_name}' already exists")
        print("Using existing Feature Group...")
    else:
        print(f"[ERROR] Error creating Feature Group: {e}")
        raise

In [ ]:
# Wait for Feature Group to be created
import time

print("Waiting for Feature Group to be created...")
status = feature_group.describe().get('FeatureGroupStatus')
print(f"Initial status: {status}")

while status == 'Creating':
    print("  Status: Creating... (waiting 15 seconds)")
    time.sleep(15)
    status = feature_group.describe().get('FeatureGroupStatus')

if status == 'Created':
    print(f"\n[OK] Feature Group is ready: {status}")
else:
    print(f"\n[WARNING] Unexpected status: {status}")

### 3.4 Ingest Training Data into Feature Store

In [ ]:
# Ingest training data into Feature Store
print(f"Ingesting {len(feature_store_df)} training records into Feature Store...")
print("This may take several minutes...\n")

start_time = time()

try:
    feature_group.ingest(
        data_frame=feature_store_df,
        max_workers=3,
        wait=True
    )
    
    elapsed_time = time() - start_time
    print(f"\n[OK] Ingestion completed in {elapsed_time:.1f} seconds")
    print(f"  Average: {elapsed_time / len(feature_store_df) * 1000:.2f} ms per record")
    
except Exception as e:
    print(f"[ERROR] Error during ingestion: {e}")
    raise

In [ ]:
# Wait for offline store to sync
print("\nWaiting for offline store synchronization...")
print("This typically takes 5-10 minutes. You can continue with other tasks.")
print("\nNote: Offline store data will be available in:")
print(f"  {s3_feature_store_prefix}/emotion-features/{feature_group.name}/")

## Phase 4: Feature Store Testing & Validation

### 4.1 Online Store Testing

In [ ]:
# Test online store retrieval
print("Testing online store retrieval...\n")

# Get SageMaker FeatureStore runtime client
featurestore_runtime = boto3.client('sagemaker-featurestore-runtime', region_name=region)

# Test retrieving a few records
test_record_ids = feature_store_df['record_id'].head(5).tolist()

for record_id in test_record_ids:
    try:
        start = time()
        response = featurestore_runtime.get_record(
            FeatureGroupName=feature_group_name,
            RecordIdentifierValueAsString=record_id
        )
        latency = (time() - start) * 1000
        
        print(f"Record {record_id}:")
        print(f"  Latency: {latency:.2f} ms")
        print(f"  Features retrieved: {len(response['Record'])}")
        
        # Show a few feature values
        for feature in response['Record'][:3]:
            print(f"    {feature['FeatureName']}: {feature['ValueAsString']}")
        print()
        
    except Exception as e:
        print(f"  [ERROR] Error retrieving record {record_id}: {e}\n")

print("[OK] Online store testing complete")

### 4.2 Offline Store Testing

In [ ]:
# Query offline store using Athena
# Note: This requires offline store to be synced (takes 5-10 minutes after ingestion)
print("Testing offline store via Athena query...\n")

try:
    # Build Athena query
    query_string = f"""
    SELECT record_id, label, meanfreq, sd, median, event_time
    FROM "{feature_group.name}"
    WHERE NOT is_deleted
    LIMIT 10
    """
    
    print(f"Query: {query_string.strip()}\n")
    
    # Execute query
    query_results = feature_group.athena_query().run(
        query_string=query_string,
        output_location=f"{s3_feature_store_prefix}/athena-results/"
    )
    
    # Wait for query completion
    query_results.wait()
    
    # Load results as dataframe
    df_results = query_results.as_dataframe()
    
    print(f"Query returned {len(df_results)} rows\n")
    print(df_results)
    
    print("\n[OK] Offline store query successful")
    
except Exception as e:
    if 'FAILED' in str(e) or 'does not exist' in str(e):
        print("[WARNING] Offline store not yet available (data sync may still be in progress)")
        print("Wait 5-10 minutes after ingestion for offline store to sync")
    else:
        print(f"[ERROR] Error querying offline store: {e}")

### 4.3 Feature Group Metadata

In [ ]:
# Display Feature Group metadata
fg_description = feature_group.describe()

print("Feature Group Metadata:")
print(f"  Name: {fg_description['FeatureGroupName']}")
print(f"  ARN: {fg_description['FeatureGroupArn']}")
print(f"  Status: {fg_description['FeatureGroupStatus']}")
print(f"  Record Identifier: {fg_description['RecordIdentifierFeatureName']}")
print(f"  Event Time Feature: {fg_description['EventTimeFeatureName']}")
print(f"  Creation Time: {fg_description['CreationTime']}")

if 'OnlineStoreConfig' in fg_description:
    print(f"\n  Online Store: Enabled")
    
if 'OfflineStoreConfig' in fg_description:
    print(f"  Offline Store: Enabled")
    print(f"    S3 URI: {fg_description['OfflineStoreConfig']['S3StorageConfig']['S3Uri']}")

## Phase 5: Save Artifacts

In [ ]:
# Save StandardScaler for production use
print("Saving feature engineering artifacts...\n")

# Save scaler
with open('/tmp/standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

scaler_s3_path = f"{s3_artifacts_prefix}/standard_scaler.pkl"
s3_client.upload_file('/tmp/standard_scaler.pkl', bucket, scaler_s3_path.replace(f"s3://{bucket}/", ""))
print(f"StandardScaler saved to: {scaler_s3_path}")

In [ ]:
# Save LabelEncoder
with open('/tmp/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

encoder_s3_path = f"{s3_artifacts_prefix}/label_encoder.pkl"
s3_client.upload_file('/tmp/label_encoder.pkl', bucket, encoder_s3_path.replace(f"s3://{bucket}/", ""))
print(f"LabelEncoder saved to: {encoder_s3_path}")

In [ ]:
# Save feature metadata
feature_metadata = {
    'feature_group_name': feature_group_name,
    'feature_group_arn': fg_description['FeatureGroupArn'],
    'original_features': original_features,
    'derived_features': derived_features,
    'features_to_scale': features_to_scale,
    'label_mapping': label_mapping,
    'num_classes': len(label_mapping),
    'total_samples': len(df_scaled),
    'train_samples': len(train),
    'test_samples': len(test),
    'validation_samples': len(validation),
    'production_samples': len(production),
    'random_seed': RANDOM_SEED,
    'creation_timestamp': datetime.now().isoformat()
}

with open('/tmp/feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

metadata_s3_path = f"{s3_artifacts_prefix}/feature_metadata.json"
s3_client.upload_file('/tmp/feature_metadata.json', bucket, metadata_s3_path.replace(f"s3://{bucket}/", ""))
print(f"Feature metadata saved to: {metadata_s3_path}")

## Summary and Next Steps

In [ ]:
# Print summary
print("="*80)
print("FEATURE ENGINEERING AND FEATURE STORE SETUP - COMPLETE")
print("="*80)

print("\nData Summary: Data Summary:")
print(f"  Total samples: {len(df_scaled):,}")
print(f"  Features: {len(features_to_scale)} (20 original + 3 derived)")
print(f"  Classes: {len(label_mapping)}")

print("\nData Splits: Data Splits:")
print(f"  Training: {len(train):,} samples (40%)")
print(f"  Test: {len(test):,} samples (10%)")
print(f"  Validation: {len(validation):,} samples (10%)")
print(f"  Production: {len(production):,} samples (40%)")

print("\nFeature Store: Feature Store:")
print(f"  Feature Group: {feature_group_name}")
print(f"  Status: {fg_description['FeatureGroupStatus']}")
print(f"  Records ingested: {len(feature_store_df):,}")
print(f"  Online store: Enabled")
print(f"  Offline store: Enabled")

print("\nS3 Locations: S3 Locations:")
print(f"  Features: {s3_features_prefix}")
print(f"  Artifacts: {s3_artifacts_prefix}")
print(f"  Feature Store: {s3_feature_store_prefix}")

print("\nNext Steps: Next Steps:")
print("  1. Model Training: Use training data from Feature Store")
print("  2. Model Evaluation: Use test/validation sets")
print("  3. Model Deployment: Deploy trained model to SageMaker endpoint")
print("  4. Production Inference: Use production batches for monitoring")
print("  5. Model Monitoring: Track data drift and model performance")
print("\n" + "="*80)